# Chapter 10: EU AI Act and NIST — Engineering Artifacts, Incident Response, and Regulatory Notification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RudrenduPaul/hardening-llm-systems-production/blob/main/companion-code/ch10-eu-ai-act-nist-engineering-artifacts/ch10_notebook.ipynb)

**Book**: *Hardening LLM Systems in Production*, Manning Books
**Author**: Rudrendu Paul

This notebook demonstrates every compliance automation component from Chapter 10:

1. Annex IV artifact index generator (Listing 10.0)
2. `AnnexIVPackage` dataclass with `completeness_score()` (Listing 10.1)
3. Annex IV CI gate — `check_annex_iv_completeness()` (Listing 10.2)
4. Output provenance recorder with HMAC-SHA256 signatures (Listing 10.3)
5. `TamperEvidentAuditLog` with chained-hash tamper detection (Listing 10.4)
6. NIST AI 600-1 triage report generator (Listing 10.4b)
7. `NISTAI6001Tracker` with `ImplementationStatus` enum and `gap_report()` (Listing 10.5)
8. Dual-framework mapping report — EU AI Act + NIST cross-reference (Listing 10.6)
9. `PostMarketMonitoringReport` dataclass (Listing 10.9)
10. Merge-blocking CI/CD gate with version and freshness checks (Listing 10.12)

**Dependencies**: `pyyaml>=6.0,<7.0` (all others are stdlib)
**Run**: `pip install pyyaml>=6.0,<7.0` then execute cells top-to-bottom.


## Manuscript reference

This notebook demonstrates the concepts from Chapter 10 of *Hardening LLM Systems in Production* (Manning, 2026).

| Notebook section | Manuscript listing | Class / function |
|------------------|--------------------|------------------|
| Annex IV artifact index | Listing 10.0 | `generate_annex_iv_index` |
| Annex IV package builder | Listing 10.1 | `AnnexIVPackage` |
| Annex IV completeness gate | Listing 10.2 | `check_annex_iv_completeness` |
| Output provenance recorder | Listing 10.3 | `create_provenance_record` / `verify_provenance_record` |
| Tamper-evident audit log | Listing 10.4 | `TamperEvidentAuditLog` |
| NIST AI 600-1 triage report | Listing 10.4b | `NistTriageReport` |
| NIST AI 600-1 tracker | Listing 10.5 | `NISTAI6001Tracker` |
| Dual-framework report generator | Listing 10.6 | `generate_dual_framework_report` |
| Post-market monitoring report | Listing 10.9 | `PostMarketMonitoringReport` |
| Incident escalation | Listing 10.7 | `IncidentEscalation` |
| Article 73 notification package | Listing 10.8 | `Article73NotificationPackage` |
| Merge-blocking CI/CD gate | Listing 10.12 | `annex_iv_ci_gate` |


In [ ]:
# ── Colab setup ────────────────────────────────────────────────────────────
# This cell only runs when executed in Google Colab.
# Local Jupyter users: skip — all code is stdlib or pip-installable.
import sys, os

if 'google.colab' in sys.modules:
    !git clone -q https://github.com/RudrenduPaul/hardening-llm-systems-production.git
    os.chdir('hardening-llm-systems-production/companion-code/ch10-eu-ai-act-nist-engineering-artifacts')
    !pip install -q pyyaml>=6.0,<7.0
    print('Colab setup complete — repo cloned, packages installed.')


In [ ]:
# Install pinned dependencies (run once per environment)
# Uncomment and run in Colab or a fresh virtual environment:
# !pip install pyyaml>=6.0.1,<7.0


## 0. Imports and setup

In [ ]:
import sys
import json
import hashlib
import hmac
import os
import time
import tempfile
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Optional

import yaml

# Import companion script (assumes notebook is in the same folder as ch10_scripts.py)
sys.path.insert(0, str(Path('.').resolve()))
from ch10_scripts import (
    ANNEX_IV_ARTIFACT_MAP,
    generate_annex_iv_index,
    ANNEX_IV_REQUIRED_FIELDS,
    ANNEX_IV_ARTIFACT_PATH_FIELDS,
    AnnexIVPackage,
    check_annex_iv_completeness,
    ProvenanceRecord,
    create_provenance_record,
    verify_provenance_record,
    TamperEvidentAuditLog,
    ClusterPriority,
    NistAction,
    NistTriageReport,
    ImplementationStatus,
    NISTControl,
    NISTAI6001Tracker,
    ANNEX_IV_TO_NIST_MAPPING,
    generate_dual_framework_report,
    PostMarketMonitoringReport,
    annex_iv_ci_gate,
)

print('All imports successful.')
print(f'Python {sys.version}')


In [ ]:
tmpdir = Path(tempfile.mkdtemp(prefix='ch10_prov_'))

# Seed placeholder artifact files that the index generator and Annex IV
# package check for on disk.
for name in ('red-team-report.pdf', 'bias-assessment.md',
             'evaluation-results.json', 'adversarial-robustness.json'):
    (tmpdir / name).write_text('placeholder artifact content')

print(f'Working directory: {tmpdir}')


## 1. Annex IV artifact index (Listing 10.0)

`generate_annex_iv_index()` checks a deployment's artifact paths against `ANNEX_IV_ARTIFACT_MAP`
and returns the sprint backlog: which of the six Annex IV sections have missing or stale artifacts.


In [ ]:
deployment_config = {
    'red_team_report_path': str(tmpdir / 'red-team-report.pdf'),
    'evaluation_results_path': str(tmpdir / 'evaluation-results.json'),
}
index = generate_annex_iv_index(deployment_config)

print(f"Ready for audit: {index['ready_for_audit']}")
print(f"Sections with gaps: {index['sections_with_gaps']}")
print(f"Sections with stale artifacts: {index['sections_with_stale_artifacts']}")


## 2. AnnexIVPackage, EU AI Act Article 11 / Annex IV documentation bundle (Listing 10.1)

The EU AI Act requires high-risk AI systems to maintain a structured **Annex IV technical
documentation** bundle. `AnnexIVPackage` captures the six engineering-facing Annex IV
sections as structured fields. `completeness_score()` checks two distinct things: whether
the required fields (`ANNEX_IV_REQUIRED_FIELDS`) are populated, and whether the referenced
artifact files (`ANNEX_IV_ARTIFACT_PATH_FIELDS`) actually exist on disk.


In [ ]:
# Build a fully populated package for a hypothetical high-risk AI system
pkg = AnnexIVPackage(
    system_name='CustomerCareBot',
    system_version='2.1.0',
    intended_purpose='Automated tier-1 customer support for retail banking',
    deployment_date='2026-01-15',
    operator_name='Acme Financial AI Ltd.',
    model_family='GPT-4',
    model_version='2024-08-06',
    training_data_description='Fine-tuned on 2.3M anonymized support transcripts (2021-2023).',
    architecture_description='RAG pipeline over policy documents with a GPT-4 generation layer.',
    components=['retrieval-service', 'generation-service', 'guardrail-filter'],
    monitoring_metrics=['hallucination_rate', 'pii_detection_rate', 'latency_p99'],
    alert_thresholds={'hallucination_rate': 0.05, 'pii_detection_rate': 0.01},
    human_oversight_design='Agents can escalate any conversation; override available at all times.',
    risk_categories_addressed=['Confabulation', 'Data Privacy', 'Human-AI Config'],
    red_team_report_path=str(tmpdir / 'red-team-report.pdf'),
    bias_assessment_path=str(tmpdir / 'bias-assessment.md'),
    adversarial_robustness_path=str(tmpdir / 'adversarial-robustness.json'),
    evaluation_results_path=str(tmpdir / 'evaluation-results.json'),
)
print(f'Completeness score: {pkg.completeness_score():.2%}')
print(f'Missing required fields: {pkg.missing_required_fields()}')
print(f'Missing or stale artifacts: {pkg.missing_or_stale_artifacts()}')


In [ ]:
# YAML serialisation — what gets stored in version control
yaml_dump = pkg.to_yaml()
print(yaml_dump[:800])


In [ ]:
# Demonstrate round-trip: YAML -> AnnexIVPackage
pkg_restored = AnnexIVPackage.from_yaml(yaml_dump)
assert pkg_restored.system_name == pkg.system_name
assert abs(pkg_restored.completeness_score() - pkg.completeness_score()) < 1e-9
print('Round-trip OK: restored package matches original.')


In [ ]:
# Incomplete package — simulate a package missing several required fields
incomplete_pkg = AnnexIVPackage(
    system_name='DraftBot',
    system_version='0.1.0',
    intended_purpose='Internal draft — not yet scoped',
    deployment_date='',
    operator_name='',
    model_family='',
    model_version='',
    training_data_description='',
    architecture_description='',
    human_oversight_design='',
)
print(f'Completeness score: {incomplete_pkg.completeness_score():.2%}')
print(f'Missing required fields: {incomplete_pkg.missing_required_fields()}')


## 3. Annex IV CI Gate (Listing 10.2)

In a real CI/CD pipeline, `check_annex_iv_completeness()` calls `sys.exit(1)` when the
documentation bundle is incomplete or its referenced artifacts are missing from disk.


In [ ]:
# PASS — complete package
print('--- CI Gate: complete package ---')
package_path = tmpdir / 'annex-iv-package.json'
pkg.to_json(package_path)
check_annex_iv_completeness(str(package_path))


In [ ]:
# FAIL — incomplete package (sys.exit(1) would stop a real pipeline here;
# we catch SystemExit so the notebook keeps running).
print('--- CI Gate: incomplete package ---')
incomplete_path = tmpdir / 'incomplete-annex-iv-package.json'
incomplete_pkg.to_json(incomplete_path)
try:
    check_annex_iv_completeness(str(incomplete_path))
except SystemExit as exc:
    print(f'Gate exited with code {exc.code} (expected in CI).')


## 4. Output Provenance Recorder (HMAC-SHA256) (Listing 10.3)

Every LLM output must be cryptographically linked to its input context, model version, and
runtime parameters. `create_provenance_record()` stores only hashes of the input and output,
never the raw text, and signs the record with HMAC-SHA256 so any retroactive edit is detectable.


In [ ]:
record = create_provenance_record(
    model_id='gpt-4o',
    model_version='2024-08-06',
    prompt_template_version='v3.1',
    session_id='sess-abc123',
    user_input='What is my account balance?',
    model_output='Your current balance is EUR 1,240.00.',
    signing_key=b'dev-only-signing-key',
)
print(f'Record ID: {record.record_id}')
print(f'Input hash: {record.input_hash}')
print(f'Output hash: {record.output_hash}')
print(f'Signature valid: {verify_provenance_record(record, signing_key=b"dev-only-signing-key")}')


In [ ]:
# Demonstrate tamper detection — modify output_hash after the fact
record.output_hash = 'a' * 64
print(f'Signature valid (after tamper): {verify_provenance_record(record, signing_key=b"dev-only-signing-key")}')


In [ ]:
# Persist provenance records to a JSONL log for retrieval by session/timestamp
from ch10_scripts import ProvenanceLog

prov_log = ProvenanceLog(tmpdir / 'provenance.jsonl')
record.output_hash = hashlib.sha256(b'restored').hexdigest()  # restore something valid
prov_log.append(record)

for i in range(3):
    r = create_provenance_record(
        model_id='gpt-4o',
        model_version='2024-08-06',
        prompt_template_version='v3.1',
        session_id=f'sess-{i:03d}',
        user_input=f'Question {i}',
        model_output=f'Answer {i}',
        signing_key=b'dev-only-signing-key',
    )
    prov_log.append(r)

loaded = prov_log.load_all()
print(f'Records on disk: {len(loaded)}')


## 5. TamperEvidentAuditLog, chained-hash append-only log (Listing 10.4)

Each log entry's hash incorporates the previous entry's hash. Any deletion, insertion, or
modification of a historical entry breaks the chain, and `verify_integrity()` reports exactly
which entry broke it.


In [ ]:
log_path = tmpdir / 'audit.jsonl'
audit_log = TamperEvidentAuditLog(str(log_path))

entries = [
    ('model_deployment', {'version': '2.1.0', 'deployed_by': 'mlops-pipeline'}),
    ('policy_update', {'policy': 'rate_limit', 'new_value': 100}),
    ('incident_detected', {'severity': 'low', 'description': 'Unusual prompt pattern'}),
]
for event_type, payload in entries:
    audit_log.append(event_type, payload)

print(f'Entries written: {len(entries)}')


In [ ]:
# Verify integrity on the clean log
print('--- Integrity check (clean log) ---')
result = audit_log.verify_integrity()
print(result)


In [ ]:
# Simulate tampering: read all lines, corrupt the payload of the middle entry, write back
lines = log_path.read_text().splitlines()
tampered_entry = json.loads(lines[1])
tampered_entry['payload'] = {'policy': 'rate_limit', 'new_value': 999999}  # tampered
lines[1] = json.dumps(tampered_entry, sort_keys=True)
log_path.write_text('\n'.join(lines) + '\n')

print('--- Integrity check (tampered log) ---')
print(audit_log.verify_integrity())


## 6. NIST AI 600-1 triage report (Listing 10.4b)

`NistTriageReport` prioritizes the 20 NIST AI 600-1 actions from section 10.5.1 by cluster
(output validation, human oversight, monitoring, data governance). `gap_summary()` is the
sprint backlog in machine-readable form.


In [ ]:
triage = NistTriageReport(deployment_type='rag', risk_level='high')
triage.add_action(NistAction(
    action_id='output-validation-1', cluster='Output validation',
    description='Maintain a golden evaluation dataset',
    engineering_artifact='Golden eval dataset + CI job',
    book_reference='Ch 2',
))
triage.add_action(NistAction(
    action_id='human-oversight-1', cluster='Human oversight design',
    description='Build a kill switch reachable within five minutes',
    engineering_artifact='Kill switch + on-call runbook',
    book_reference='Section 10.3.1',
))
triage.add_action(NistAction(
    action_id='monitoring-1', cluster='Monitoring and alerting',
    description='Track hallucination rate as a time-series metric',
    engineering_artifact='Hallucination rate dashboard',
    book_reference='Section 10.5.1',
))
triage.mark_implemented('output-validation-1', evidence_path=str(tmpdir / 'evaluation-results.json'))

summary = triage.gap_summary()
print(f"Implemented: {summary['implemented_count']}/{summary['total_actions']}")
print(f"Gaps: {[g['action_id'] for g in summary['gaps']]}")


## 7. NIST AI 600-1 Tracker (Listing 10.5)

`NISTAI6001Tracker` tracks per-control implementation status, owner, and evidence path.
`gap_report()` returns a coverage percentage plus the list of controls not yet implemented
or verified — exactly what you hand to legal or executive stakeholders.


In [ ]:
tracker = NISTAI6001Tracker()
tracker.add_control(NISTControl(control_id='GV-1.1', risk_category='Govern',
                                 description='Policies for organizational TEVV'))
tracker.add_control(NISTControl(control_id='GV-1.2', risk_category='Govern',
                                 description='Organizational commitment to AI risk governance'))
tracker.add_control(NISTControl(control_id='MS-1.1', risk_category='Measure',
                                 description='AI system risks are identified and assessed'))
tracker.add_control(NISTControl(control_id='MG-4.1', risk_category='Manage',
                                 description='Post-deployment risks are monitored'))

tracker.update_status('GV-1.1', ImplementationStatus.VERIFIED,
                       owner='AI Governance Team', evidence_path='governance-policy-v3.pdf')
tracker.update_status('GV-1.2', ImplementationStatus.IMPLEMENTED,
                       owner='CTO Office', evidence_path='board-charter.pdf')
tracker.update_status('MS-1.1', ImplementationStatus.IN_PROGRESS, owner='Risk Team')

print('Controls tracked:', list(tracker.controls.keys()))


In [ ]:
gap_report = tracker.gap_report()
print(f"Generated at        : {gap_report['generated_at']}")
print(f"Coverage percentage : {gap_report['coverage_percentage']}%")
print(f"Summary             : {gap_report['summary']}")
print(f"Gap count           : {gap_report['gap_count'] if 'gap_count' in gap_report else len(gap_report['gaps'])}")
print()
print('Open gaps:')
for g in gap_report['gaps']:
    print(f"  - {g['control_id']}: {g['description']} (status={g['status']})")


In [ ]:
# Export tracker state to JSON for audit hand-off
tracker_path = tmpdir / 'nist-tracker.json'
tracker.to_json(tracker_path)
print(f'Tracker exported to: {tracker_path.name}')
print(tracker_path.read_text()[:400])


## 8. Dual-Framework Mapping Report (Listing 10.6)

Many compliance teams manage EU AI Act and NIST AI RMF in separate silos.
`generate_dual_framework_report()` produces a single JSON report that cross-references
`ANNEX_IV_TO_NIST_MAPPING` against the current Annex IV completeness and NIST coverage,
including an `overall_status` field (COMPLIANT requires 100% Annex IV completeness and
at least 80% NIST coverage — an engineering threshold, never a legal determination).


In [ ]:
report_path = tmpdir / 'compliance-report.json'
report_json = generate_dual_framework_report(pkg, tracker, str(report_path))
report = json.loads(report_json)

print('--- Report summary ---')
print(f"System: {report['system_name']} v{report['system_version']}")
print(f"Overall status: {report['overall_status']}")
print(f"Annex IV score: {report['annex_iv_completeness']['score']:.2%}")
print(f"NIST coverage: {report['nist_coverage']['coverage_percentage']}%")


In [ ]:
print('--- EU AI Act / NIST cross-reference (Annex IV point -> NIST categories) ---')
for point, categories in list(ANNEX_IV_TO_NIST_MAPPING.items())[:3]:
    print(f'  {point}: {categories}')


## 9. PostMarketMonitoringReport (Listing 10.9)

EU AI Act Article 72 requires high-risk AI providers to run a systematic post-market
monitoring program and document its results. `PostMarketMonitoringReport` captures traffic,
quality metrics, and incident counts for a reporting period, and flags whether any P0
incident requires Article 73 regulatory notification.


In [ ]:
pmm = PostMarketMonitoringReport(
    system_name='CustomerCareBot',
    system_version='2.1.0',
    reporting_period_start='2026-07-01',
    reporting_period_end='2026-07-31',
    total_queries=148_320,
    hallucination_rate=0.031,
    pii_detection_rate=0.004,
    bias_gap_max=0.06,
    incidents_p0=0,
    incidents_p1=1,
    incidents_p2=3,
    serious_incidents_reported=0,
    trend_alerts=['hallucination rate stable over trailing 30 days'],
)
print(f'Requires regulatory notification: {pmm.requires_regulatory_notification()}')


In [ ]:
# Simulate a serious-incident scenario
pmm_serious = PostMarketMonitoringReport(
    system_name='CustomerCareBot',
    system_version='2.1.0',
    reporting_period_start='2026-08-01',
    reporting_period_end='2026-08-31',
    total_queries=151_002,
    hallucination_rate=0.045,
    pii_detection_rate=0.011,
    bias_gap_max=0.09,
    incidents_p0=1,
    incidents_p1=2,
    incidents_p2=4,
    serious_incidents_reported=1,
    trend_alerts=['bias gap widened beyond threshold for gender-associated occupations'],
)
print(f'Requires regulatory notification: {pmm_serious.requires_regulatory_notification()}')


In [ ]:
# Persist to JSON
pmm_path = tmpdir / 'pmm-report-2026-07.json'
pmm.to_json(str(pmm_path))
print(f'Report saved to: {pmm_path.name}')

saved = json.loads(pmm_path.read_text())
print(f"Keys: {list(saved['monitoring_report'].keys())}")


## 10. Merge-blocking CI/CD gate (Listing 10.12)

`annex_iv_ci_gate()` is the gate wired into the unified PR-gate pipeline from chapter 1 as
merge-blocking signal #10. It checks that the Annex IV package matches the model version
being deployed, that all required fields are present, and that referenced artifacts are
both on disk and not stale.


In [ ]:
print('--- CI/CD gate: version matches, artifacts fresh ---')
annex_iv_ci_gate(str(package_path), current_model_version=pkg.system_version)


In [ ]:
print('--- CI/CD gate: version mismatch (would block the merge) ---')
try:
    annex_iv_ci_gate(str(package_path), current_model_version='9.9.9')
except SystemExit as exc:
    print(f'Gate exited with code {exc.code} (expected — blocks the merge).')


## 11. End-to-end compliance pipeline summary

This cell pulls together all components to show how they connect in a production compliance
workflow.


In [ ]:
print('=== Chapter 10 — Compliance Pipeline Summary ===')
print()

checks = [
    ('AnnexIVPackage completeness', f'{pkg.completeness_score():.2%}', pkg.completeness_score() >= 1.0),
    ('NIST AI 600-1 coverage', f"{gap_report['coverage_percentage']}%",
     gap_report['coverage_percentage'] >= 80.0),
    ('Dual-framework status', report['overall_status'], report['overall_status'] == 'COMPLIANT'),
    ('Audit log integrity', audit_log.verify_integrity()['status'],
     audit_log.verify_integrity()['status'] == 'intact'),
    ('Post-market notification required', pmm.requires_regulatory_notification(),
     not pmm.requires_regulatory_notification()),
]

for label, value, passed in checks:
    marker = 'PASS' if passed else 'REVIEW'
    print(f'  [{marker}] {label}: {value}')


## Cleanup

In [ ]:
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)
print(f'Temp directory removed: {tmpdir}')
